In [12]:
import pandas as pd
import numpy as np
import os

In [13]:
data = """
Rk,Squad,Country,LgRk,MP,W,D,L,GF,GA,GD,Pts,Pts/MP,xG,xGA,xGD,xGD/90,Attendance,Top Team Scorer,Goalkeeper
1,Paris S-G,fr FRA,1,34,26,6,2,92,35,+57,84,2.47,88.9,31.2,+57.7,+1.70,47413,Ousmane Dembélé - 21,Gianluigi Donnarumma
2,Bayern Munich,de GER,1,34,25,7,2,99,32,+67,82,2.41,81.7,25.5,+56.2,+1.65,75000,Harry Kane - 26,Manuel Neuer
3,Barcelona,es ESP,1,38,28,4,6,102,39,+63,88,2.32,91.5,41.9,+49.5,+1.30,45953,Robert Lewandowski - 27,Iñaki Peña
4,Liverpool,eng ENG,1,38,25,9,4,86,41,+45,84,2.21,82.2,38.6,+43.6,+1.15,60324,Mohamed Salah - 29,Alisson
5,Real Madrid,es ESP,2,38,26,6,6,78,38,+40,84,2.21,75.3,42.8,+32.5,+0.86,69807,Kylian Mbappé - 31,Thibaut Courtois
6,Napoli,it ITA,1,38,24,10,4,59,27,+32,82,2.16,52.9,29.6,+23.3,+0.61,50909,Romelu Lukaku - 14,Alex Meret
7,Inter,it ITA,2,38,24,9,5,79,35,+44,81,2.13,66.9,36.4,+30.5,+0.80,70126,Marcus Thuram - 14,Yann Sommer
8,Leverkusen,de GER,2,34,19,12,3,72,43,+29,69,2.03,56.9,36.2,+20.6,+0.61,29961,Patrik Schick - 21,Lukáš Hrádecký
9,Atlético Madrid,es ESP,3,38,22,10,6,68,30,+38,76,2.00,64.6,33.4,+31.2,+0.82,60883,Alexander Sørloth - 20,Jan Oblak
10,Atalanta,it ITA,3,38,22,8,8,78,37,+41,74,1.95,65.0,38.6,+26.4,+0.69,22675,Mateo Retegui - 25,Marco Carnesecchi
11,Arsenal,eng ENG,2,38,20,14,4,69,34,+35,74,1.95,59.9,34.4,+25.5,+0.67,60251,Kai Havertz - 9,David Raya
12,Marseille,fr FRA,2,34,20,5,9,74,47,+27,65,1.91,63.8,45.2,+18.6,+0.55,63507,Mason Greenwood - 21,Gerónimo Rulli
13,Manchester City,eng ENG,3,38,21,8,9,72,44,+28,71,1.87,68.1,47.7,+20.4,+0.54,52756,Erling Haaland - 22,Ederson
14,Athletic Club,es ESP,4,38,19,13,6,54,29,+25,70,1.84,53.0,37.8,+15.2,+0.40,48420,Oihan Sancet - 15,Unai Simón
15,Juventus,it ITA,4,38,18,16,4,58,35,+23,70,1.84,51.0,34.0,+16.9,+0.45,40275,Dušan Vlahović - 10,Michele Di Gregorio
16,Villarreal,es ESP,5,38,20,10,8,71,51,+20,70,1.84,64.8,44.4,+20.4,+0.54,18266,Ayoze Pérez - 19,Diego Conde
17,Chelsea,eng ENG,4,38,20,9,9,64,43,+21,69,1.82,67.8,47.3,+20.5,+0.54,39672,Cole Palmer - 15,Robert Sánchez
18,Roma,it ITA,5,38,20,9,9,56,35,+21,69,1.82,53.0,40.9,+12.1,+0.32,61895,Artem Dovbyk - 12,Mile Svilar
19,Monaco,fr FRA,3,34,18,7,9,63,41,+22,61,1.79,72.8,34.8,+38.0,+1.12,12164,Mika Biereth - 13,Philipp Köhn
20,Nice,fr FRA,4,34,17,9,8,66,41,+25,60,1.76,61.6,40.2,+21.4,+0.63,24299,Evann Guessand - 12,Marcin Bułka
21,Eint Frankfurt,de GER,3,34,17,9,8,68,46,+22,60,1.76,65.1,48.0,+17.1,+0.50,57600,Omar Marmoush and Hugo Ekitike - 15,Kevin Trapp
22,Lille,fr FRA,5,34,17,9,8,52,36,+16,60,1.76,54.7,42.5,+12.2,+0.36,42417,Jonathan David - 16,Lucas Chevalier
23,Newcastle Utd,eng ENG,5,38,20,6,12,68,47,+21,66,1.74,63.8,45.5,+18.3,+0.48,52187,Alexander Isak - 23,Nick Pope
24,Aston Villa,eng ENG,6,38,19,9,10,58,51,+7,66,1.74,56.1,50.1,+5.9,+0.16,42079,Ollie Watkins - 16,Emiliano Martínez
25,Fiorentina,it ITA,6,38,19,8,11,60,41,+19,65,1.71,48.9,42.8,+6.1,+0.16,20358,Moise Kean - 19,David de Gea
26,Nott'ham Forest,eng ENG,7,38,19,8,11,58,46,+12,65,1.71,45.5,48.9,-3.4,-0.09,30059,Chris Wood - 20,Matz Sels
27,Lazio,it ITA,7,38,18,11,9,61,49,+12,65,1.71,57.2,37.7,+19.5,+0.51,41161,Pedro and Valentín Castellanos - 10,Ivan Provedel
28,Dortmund,de GER,4,34,17,6,11,71,51,+20,57,1.68,61.2,42.8,+18.4,+0.54,81365,Serhou Guirassy - 21,Gregor Kobel
29,Lyon,fr FRA,6,34,17,6,11,65,46,+19,57,1.68,60.3,45.7,+14.6,+0.43,51191,Alexandre Lacazette - 15,Lucas Perri
30,Strasbourg,fr FRA,7,34,16,9,9,56,44,+12,57,1.68,49.3,56.1,-6.9,-0.20,19378,Emanuel Emegha - 14,Đorđe Petrović
31,Milan,it ITA,8,38,18,9,11,61,43,+18,63,1.66,63.7,41.3,+22.4,+0.59,71523,Christian Pulisic - 11,Mike Maignan
32,Bologna,it ITA,9,38,16,14,8,57,47,+10,62,1.63,49.3,36.7,+12.6,+0.33,27772,Riccardo Orsolini - 15,Łukasz Skorupski
33,Freiburg,de GER,5,34,16,7,11,49,53,-4,55,1.62,44.3,42.4,+1.8,+0.05,34188,Ritsu Doan - 10,Noah Atubolu
34,Brighton,eng ENG,8,38,16,13,9,66,59,+7,61,1.61,58.7,54.6,+4.1,+0.11,31798,Danny Welbeck and Kaoru Mitoma - 10,Bart Verbruggen
35,Betis,es ESP,6,38,16,12,10,57,50,+7,60,1.58,54.7,50.6,+4.1,+0.11,51542,Isco - 9,Adrián
36,Mainz 05,de GER,6,34,14,10,10,55,43,+12,52,1.53,50.1,48.1,+2.0,+0.06,32354,Jonathan Burkardt - 18,Robin Zentner
37,Lens,fr FRA,8,34,15,7,12,42,39,+3,52,1.53,51.8,45.3,+6.5,+0.19,37971,Neil El Aynaoui - 8,Brice Samba
38,RB Leipzig,de GER,7,34,13,12,9,53,48,+5,51,1.50,46.6,53.3,-6.8,-0.20,44803,Benjamin Šeško - 13,Péter Gulácsi
39,Werder Bremen,de GER,8,34,14,9,11,54,57,-3,51,1.50,49.1,48.1,+1.0,+0.03,41350,Jens Stage - 10,Michael Zetterer
40,Bournemouth,eng ENG,9,38,15,11,12,58,46,+12,56,1.47,64.0,48.5,+15.5,+0.41,11210,Justin Kluivert - 12,Kepa Arrizabalaga
41,Stuttgart,de GER,9,34,14,8,12,64,53,+11,50,1.47,62.3,46.9,+15.4,+0.45,59265,Ermedin Demirović - 15,Alexander Nübel
42,Brentford,eng ENG,10,38,16,8,14,66,57,+9,56,1.47,59.0,55.4,+3.6,+0.09,18742,Bryan Mbeumo - 20,Mark Flekken
43,Brest,fr FRA,9,34,15,5,14,52,59,-7,50,1.47,45.4,57.4,-12.0,-0.35,14536,Ludovic Ajorque - 13,Marco Bizot
44,Celta Vigo,es ESP,7,38,16,7,15,59,57,+2,55,1.45,54.2,43.4,+10.9,+0.29,21504,Borja Iglesias - 11,Vicente Guaita
45,Fulham,eng ENG,11,38,15,9,14,54,54,0,54,1.42,49.0,47.2,+1.8,+0.05,26826,Raúl Jiménez - 12,Bernd Leno
46,Crystal Palace,eng ENG,12,38,13,14,11,51,51,0,53,1.39,60.4,49.1,+11.3,+0.30,25064,Jean-Philippe Mateta - 14,Dean Henderson
47,Osasuna,es ESP,9,38,12,16,10,48,52,-4,52,1.37,44.0,53.7,-9.7,-0.25,20476,Ante Budimir - 21,Sergio Herrera
48,Rayo Vallecano,es ESP,8,38,13,13,12,41,45,-4,52,1.37,45.4,49.0,-3.5,-0.09,12908,Jorge de Frutos - 6,Augusto Batalla
49,Gladbach,de GER,10,34,13,6,15,55,57,-2,45,1.32,50.6,63.4,-12.7,-0.37,53056,Tim Kleindienst - 16,Moritz Nicolas
50,Como,it ITA,10,38,13,10,15,49,52,-3,49,1.29,45.4,43.0,+2.4,+0.06,10604,Assane Diao - 8,Jean Butez
51,Wolfsburg,de GER,11,34,11,10,13,56,54,+2,43,1.26,47.5,52.5,-5.1,-0.15,24660,Mohamed Amoura - 10,Kamil Grabara
52,Everton,eng ENG,13,38,11,15,12,42,44,-2,48,1.26,41.8,46.2,-4.5,-0.12,38439,Iliman Ndiaye - 9,Jordan Pickford
53,Mallorca,es ESP,10,38,13,9,16,35,44,-9,48,1.26,38.8,46.8,-8.1,-0.21,18502,Cyle Larin and Vedat Muriqi - 7,Dominik Greif
54,Augsburg,de GER,12,34,11,10,13,35,51,-16,43,1.26,34.7,48.9,-14.2,-0.42,29820,Alexis Claude-Maurice - 9,Finn Dahmen
55,Toulouse,fr FRA,10,34,11,9,14,44,43,+1,42,1.24,49.1,36.8,+12.4,+0.36,25593,Zakaria Aboukhlal - 7,Guillaume Restes
56,Auxerre,fr FRA,11,34,11,9,14,48,51,-3,42,1.24,41.0,58.6,-17.6,-0.52,16634,Gaëtan Perrin and Hamed Junior Traorè - 10,Donovan Léon
57,Rennes,fr FRA,12,34,13,2,19,51,50,+1,41,1.21,48.6,46.0,+2.6,+0.08,27970,Arnaud Kalimuendo - 17,Brice Samba
58,Valencia,es ESP,12,38,11,13,14,44,54,-10,46,1.21,43.3,51.9,-8.6,-0.23,43042,Hugo Duro - 11,Giorgi Mamardashvili
59,Real Sociedad,es ESP,11,38,13,7,18,35,46,-11,46,1.21,42.5,43.9,-1.5,-0.04,29877,Mikel Oyarzabal - 9,Álex Remiro
60,Union Berlin,de GER,13,34,10,10,14,35,51,-16,40,1.18,37.0,47.8,-10.7,-0.32,21953,Benedict Hollerbach - 9,Frederik Rønnow
61,Torino,it ITA,11,38,10,14,14,39,45,-6,44,1.16,34.6,50.4,-15.8,-0.42,23327,Che Adams - 9,Vanja Milinković-Savić
62,Udinese,it ITA,12,38,12,8,18,41,56,-15,44,1.16,38.2,52.6,-14.4,-0.38,21098,Lorenzo Lucca - 12,Maduka Okoye
63,Genoa,it ITA,13,38,10,13,15,37,49,-12,43,1.13,37.2,47.2,-10.0,-0.26,31248,Andrea Pinamonti - 10,Nicola Leali
64,West Ham,eng ENG,14,38,11,10,17,46,62,-16,43,1.13,47.0,59.7,-12.7,-0.33,62463,Jarrod Bowen - 13,Alphonse Areola
65,Getafe,es ESP,13,38,11,9,18,34,39,-5,42,1.11,36.8,46.2,-9.4,-0.25,11469,Mauro Arambarri - 10,David Soria
66,Manchester Utd,eng ENG,15,38,11,9,18,44,54,-10,42,1.11,52.6,53.8,-1.3,-0.03,73747,Bruno Fernandes and Amad Diallo - 8,André Onana
67,Alavés,es ESP,15,38,10,12,16,38,48,-10,42,1.11,42.8,47.0,-4.2,-0.11,17318,Kiké - 13,Antonio Sivera
68,Espanyol,es ESP,14,38,11,9,18,40,51,-11,42,1.11,34.4,53.8,-19.4,-0.51,25640,Javi Puado - 12,Joan García
69,Wolves,eng ENG,16,38,12,6,20,54,69,-15,42,1.11,43.7,58.1,-14.5,-0.38,30695,Matheus Cunha - 15,José Sá
70,Sevilla,es ESP,17,38,10,11,17,42,55,-13,41,1.08,42.7,47.4,-4.7,-0.12,35619,Dodi Lukebakio - 11,Ørjan Nyland
71,Girona,es ESP,16,38,11,8,19,44,60,-16,41,1.08,42.5,50.4,-8.0,-0.21,11657,Cristhian Stuani - 11,Paulo Gazzaniga
72,Nantes,fr FRA,13,34,8,12,14,39,52,-13,36,1.06,39.1,55.4,-16.3,-0.48,30317,Matthis Abline - 9,Anthony Lopes
73,Angers,fr FRA,14,34,10,6,18,32,53,-21,36,1.06,36.0,57.8,-21.8,-0.64,13142,Esteban Lepaul - 9,Yahia Fofana
74,Leganés,es ESP,18,38,9,13,16,39,56,-17,40,1.05,36.0,59.4,-23.4,-0.62,11135,Dani Raba - 8,Marko Dmitrović
75,Tottenham,eng ENG,17,38,11,5,22,64,65,-1,38,1.00,58.8,63.3,-4.5,-0.12,61127,Brennan Johnson - 11,Guglielmo Vicario
76,Le Havre,fr FRA,15,34,10,4,20,40,71,-31,34,1.00,41.3,62.8,-21.5,-0.63,20218,Abdoulaye Touré - 10,Arthur Desmas
77,Reims,fr FRA,16,34,8,9,17,33,47,-14,33,0.97,37.4,57.4,-19.9,-0.59,15655,Keito Nakamura - 11,Yehvann Diouf
78,Hellas Verona,it ITA,14,38,10,7,21,34,66,-32,37,0.97,33.8,53.4,-19.6,-0.51,25391,Casper Tengstedt - 6,Lorenzo Montipò
79,Parma,it ITA,16,38,7,15,16,44,58,-14,36,0.95,43.1,56.4,-13.3,-0.35,19236,Ange-Yoan Bonny - 6,Zion Suzuki
80,Cagliari,it ITA,15,38,9,9,20,40,56,-16,36,0.95,42.7,55.3,-12.6,-0.33,16071,Roberto Piccoli - 10,Elia Caprile
81,St. Pauli,de GER,14,34,8,8,18,28,41,-13,32,0.94,35.3,45.4,-10.1,-0.30,29506,Morgan Guilavogui - 6,Nikola Vasilj
82,Hoffenheim,de GER,15,34,7,11,16,46,68,-22,32,0.94,42.6,55.8,-13.1,-0.39,25309,Andrej Kramarić - 11,Oliver Baumann
83,Lecce,it ITA,17,38,8,10,20,27,58,-31,34,0.89,34.6,57.1,-22.5,-0.59,25980,Nikola Krstović - 11,Wladimiro Falcone
84,Saint-Étienne,fr FRA,17,34,8,6,20,39,77,-38,30,0.88,36.4,74.3,-37.9,-1.15,30220,Lucas Stassin - 12,Gautier Larsonneur
85,Heidenheim,de GER,16,34,8,5,21,37,64,-27,29,0.85,40.3,57.6,-17.4,-0.51,15000,Marvin Pieringer - 7,Kevin Müller
86,Las Palmas,es ESP,19,38,8,8,22,40,61,-21,32,0.84,36.3,66.8,-30.5,-0.80,23010,Fábio Silva - 10,Jasper Cillessen
87,Empoli,it ITA,18,38,6,13,19,33,59,-26,31,0.82,32.5,50.1,-17.6,-0.46,10648,Sebastiano Esposito - 8,Devis Vásquez
88,Venezia,it ITA,19,38,5,14,19,32,56,-24,29,0.76,35.8,57.5,-21.7,-0.57,10415,Joel Pohjanpalo - 6,Filip Stankovic
89,Holstein Kiel,de GER,17,34,6,7,21,49,80,-31,25,0.74,39.8,61.0,-21.1,-0.62,14893,Shuto Machino - 11,Timon Weiner
90,Bochum,de GER,18,34,6,7,21,33,67,-34,25,0.74,41.8,63.2,-21.3,-0.63,25540,Myron Boadu - 9,Patrick Drewes
91,Leicester City,eng ENG,18,38,6,7,25,33,80,-47,25,0.66,32.6,71.9,-39.3,-1.03,31448,Jamie Vardy - 9,Mads Hermansen
92,Ipswich Town,eng ENG,19,38,4,10,24,36,82,-46,22,0.58,34.4,72.7,-38.3,-1.01,29742,Liam Delap - 12,Arijanet Muric
93,Monza,it ITA,20,38,3,9,26,28,69,-41,18,0.47,29.9,54.7,-24.8,-0.65,11271,Dany Mota - 5,Stefano Turati
94,Montpellier,fr FRA,18,34,4,4,26,23,79,-56,16,0.47,33.5,63.3,-29.9,-0.91,13483,Arnaud Nordin - 4,Benjamin Lecomte
95,Valladolid,es ESP,20,38,4,4,30,26,90,-64,16,0.42,34.6,67.6,-33.0,-0.87,19831,Mamadou Sylla - 5,Karl Jakob Hein
96,Southampton,eng ENG,20,38,2,6,30,26,86,-60,12,0.32,32.7,84.8,-52.1,-1.37,30882,Paul Onuachu - 4,Aaron Ramsdale
"""

#from io import StringIO
#big5_df = pd.read_csv(StringIO(data.strip()))
#print(big5_df.head())
#big5_df.to_csv('../data/raw/big5_leagues_stats.csv', index=False)

In [14]:
# Position standardisation map — FBref uses compound codes like "MF,DF".
# We keep the PRIMARY (first) position and map it to a clean standard label.
_POSITION_MAP = {
    'GK':  'GK',
    'DF':  'DF',
    'MF':  'MF',
    'FW':  'FW',
    '"MF,FW"': 'MF/FW',
    '"MF,DF"': 'MF/DF'
}

def standardise_position(pos_str):
    """
    Convert compound FBref positions like 'MF,DF' or 'FW,MF' to a single
    standard label by taking the FIRST listed position.
    """
    if pd.isna(pos_str) or str(pos_str).strip() == '':
        return np.nan
    primary = str(pos_str).split(',')[0].strip()
    return _POSITION_MAP.get(primary, primary)

In [15]:
def clean_fbref_rows(df, league_code, season, add_rank=True):
    """
    Apply all row-level cleaning to a raw FBref summary DataFrame.

    Steps performed:
      1. Add a rank column (row number within this file).
      2. Remove rows where ANY cell contains '90s' (FBref sub-headers).
      3. Standardise positions: 'MF,DF' → 'MF', 'FW,MF' → 'FW'.
      4. Strip commas from minutes and convert to numeric.
      5. Deduplicate players (same name+age) — keep row with MOST minutes.
    """
    df = df.copy()

    # 1. ADD RANK
    if add_rank:
        df.insert(0, 'fbref_rank', range(1, len(df) + 1))

    # 2. REMOVE '90s' ROWS
    mask_90s = df.apply(
        lambda col: col.astype(str).str.strip() == '90s', axis=0
    ).any(axis=1)
    n_dropped = mask_90s.sum()
    df = df[~mask_90s].reset_index(drop=True)
    if n_dropped:
        print(f"    [clean] Dropped {n_dropped} '90s' header rows")

    # 3. STANDARDISE POSITION
    pos_col = None
    for candidate in ('Pos', 'position_fbref', 'position'):
        if candidate in df.columns:
            pos_col = candidate
            break
    if pos_col:
        df[pos_col] = df[pos_col].apply(standardise_position)

    # 4. CLEAN MINUTES (strip commas)
    min_col = None
    for candidate in ('Min', 'minutes'):
        if candidate in df.columns:
            min_col = candidate
            break
    if min_col:
        df[min_col] = (
            df[min_col].astype(str)
                       .str.replace(',', '', regex=False)
                       .str.strip()
        )
        df[min_col] = pd.to_numeric(df[min_col], errors='coerce')

    # 5. COMBINE DUPLICATE ROWS (keep row with most minutes)
    name_col = None
    for candidate in ('Player', 'name'):
        if candidate in df.columns:
            name_col = candidate
            break
    age_col = 'Age' if 'Age' in df.columns else None

    if name_col and min_col:
        group_cols = [name_col] + ([age_col] if age_col else [])
        before = len(df)
        df = (
            df.sort_values(min_col, ascending=False, na_position='last')
              .drop_duplicates(subset=group_cols, keep='first')
              .reset_index(drop=True)
        )
        after = len(df)
        if before != after:
            print(f"    [clean] Merged {before - after} duplicate rows "
                  f"(kept highest-minutes per name+age)")

    # Re-number ranks after cleaning
    if add_rank and 'fbref_rank' in df.columns:
        df['fbref_rank'] = range(1, len(df) + 1)

    return df

In [16]:
def load_fbref_summary(league_code, season):
    """
    Load summary CSV with all cleaning applied via clean_fbref_rows().
    """
    filename = f"{league_code}_{season}_summary.csv"
    path = os.path.join(FBREF_DIR, filename)
    if not os.path.exists(path):
        print(f"  [MISSING] {filename} — skipping")
        return pd.DataFrame()

    df = pd.read_csv(path)
    df = clean_fbref_table(df)
    
    # Apply all row-level cleaning BEFORE renaming
    df = clean_fbref_rows(df, league_code, season, add_rank=True)

    # Rename to standard names AFTER cleaning
    rename_map = {
        'Player': 'name',
        'Pos':    'position_fbref',
        'Club':   'club',
        'Squad':  'club',
        'MP':     'appearances',
        'Min':    'minutes',
        'G':      'goals',
        'A':      'assists',
        'Gls':    'goals',
        'Ast':    'assists',
    }
    df.rename(columns=rename_map, inplace=True, errors='ignore')

    # Ensure numeric types
    if 'minutes' in df.columns:
        df['minutes'] = pd.to_numeric(df['minutes'], errors='coerce')
    for col in ['goals', 'assists', 'appearances']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    keep = [c for c in ['fbref_rank', 'name', 'club', 'appearances', 
                        'minutes', 'goals', 'assists', 'position_fbref']
            if c in df.columns]
    return df[keep].dropna(subset=['name'])
    # Use Kaggle name where available, fall back to FBref
    merged['name'] = merged['name_kaggle'].fillna(merged['name_fbref'])
    
    # Use Kaggle position if available, fall back to standardised FBref position
    if 'position_kaggle' in merged.columns:
        merged['position'] = merged['position_kaggle'].fillna(
            merged.get('position_fbref', pd.Series(dtype=str))
        )
    elif 'position_fbref' in merged.columns:
        merged['position'] = merged['position_fbref']

In [17]:
final_columns = [
        'fbref_rank',          # NEW - rank from original FBref file
        'player_id',
        'name',
        'position',
        'season',
        'league',
        'club',
        'club_finish',
        'appearances',
        'minutes',
        'goals',
        'assists',
        'xG',
        'xA',
        'npxG',
        'goals_per_90',
        'assists_per_90',
        'xG_per_90',
        'xA_per_90',
        'market_value_eur',
        'age_at_season_end',
    ]

In [18]:
def load_player_stats_files():
    """
    Load the Big 5 league player stats CSV files (2020-21 to 2024-25).
    These are ALREADY combined across all 5 leagues in each season file.
    
    Expected files:
      data/raw/player_stats/player_stats_20-21.csv
      data/raw/player_stats/player_stats_21-22.csv
      data/raw/player_stats/player_stats_22-23.csv
      data/raw/player_stats/player_stats_23-24.csv
      data/raw/player_stats/player_stats_24-25.csv
    
    Columns: Rk, Player, Nation, Pos, Squad, Comp, Age, Born, MP, Starts, 
             Min, 90s, Gls, Ast, G+A, G-PK, PK, PKatt, CrdY, CrdR, 
             Gls_90, Ast_90, G+A_90, G-PK_90, G+A-PK_90
    
    Returns: Combined DataFrame with season column added.
    """
    STATS_DIR = os.path.join('..', 'data', 'raw', 'player_stats')
    
    # Map filenames to our standard season format
    season_map = {
        'player_stats_20-21.csv': '2020-2021',
        'player_stats_21-22.csv': '2021-2022',
        'player_stats_22-23.csv': '2022-2023',
        'player_stats_23-24.csv': '2023-2024',
        'player_stats_24-25.csv': '2024-2025',
    }
    
    all_stats = []
    
    for filename, season in season_map.items():
        path = os.path.join(STATS_DIR, filename)
        if not os.path.exists(path):
            print(f"  [MISSING] {filename} — skipping")
            continue
        
        df = pd.read_csv(path)
        
        # Clean repeated header rows
        if 'Rk' in df.columns:
            df = df[df['Rk'] != 'Rk'].copy()
        
        # Apply the same row cleaning as FBref
        df = clean_fbref_rows(df, 'stats', season, add_rank=True)
        
        # Add season column
        df['season'] = season
        
        # Rename to match our schema
        rename_map = {
            'Player': 'name',
            'Pos': 'position_stats',
            'Squad': 'club',
            'MP': 'appearances_stats',
            'Min': 'minutes_stats',
            'Gls': 'goals_stats',
            'Ast': 'assists_stats',
        }
        df.rename(columns=rename_map, inplace=True, errors='ignore')
        
        # Keep relevant columns
        keep_cols = [c for c in ['fbref_rank', 'name', 'club', 'season', 
                                  'position_stats', 'appearances_stats', 
                                  'minutes_stats', 'goals_stats', 'assists_stats',
                                  '90s', 'Gls_90', 'Ast_90', 'G+A_90', 
                                  'G-PK', 'G-PK_90', 'G+A-PK_90']
                     if c in df.columns]
        
        df = df[keep_cols]
        all_stats.append(df)
        print(f"    Loaded {filename}: {len(df):,} rows")
    
    if not all_stats:
        return pd.DataFrame()
    
    combined = pd.concat(all_stats, ignore_index=True)
    print(f"  Total player stats rows: {len(combined):,}")
    return combined

In [19]:
def load_player_ratings():
    """
    Load player ratings from league-specific folders.
    
    Expected structure:
      data/raw/ratings/BL/2021BL.csv, 2122BL.csv, 2223BL.csv, 2324BL.csv, 2425BL.csv
      data/raw/ratings/L1/2021L1.csv, 2122L1.csv, ...
      data/raw/ratings/LL/2021LL.csv, 2122LL.csv, ...
      data/raw/ratings/PL/2021PL.csv, 2122PL.csv, ...
      data/raw/ratings/SA/2021SA.csv, 2122SA.csv, ...
    
    Columns: Rank, Player, Rating
    
    Returns: DataFrame with columns [name, season, league, rating]
    """
    RATINGS_DIR = os.path.join('..', 'data', 'raw', 'ratings')
    
    # Map league codes to our standard league names
    league_map = {
        'BL': 'Bundesliga',
        'L1': 'Ligue 1',
        'LL': 'La Liga',
        'PL': 'Premier League',
        'SA': 'Serie A',
    }
    
    # Map year prefixes to our standard season format
    year_map = {
        '2021': '2020-2021',
        '2122': '2021-2022',
        '2223': '2022-2023',
        '2324': '2023-2024',
        '2425': '2024-2025',
    }
    
    all_ratings = []
    
    for league_code, league_name in league_map.items():
        league_dir = os.path.join(RATINGS_DIR, league_code)
        
        if not os.path.exists(league_dir):
            print(f"  [MISSING] ratings folder for {league_code} — skipping")
            continue
        
        for year_prefix, season in year_map.items():
            filename = f"{year_prefix}{league_code}.csv"
            path = os.path.join(league_dir, filename)
            
            if not os.path.exists(path):
                continue
            
            df = pd.read_csv(path)
            
            # Rename columns
            df.rename(columns={'Player': 'name', 'Rating': 'rating'}, 
                     inplace=True, errors='ignore')
            
            # Add season and league
            df['season'] = season
            df['league'] = league_name
            
            # Keep only what we need
            df = df[['name', 'season', 'league', 'rating']].copy()
            
            # Clean player names for matching
            df['name'] = df['name'].str.strip()
            
            all_ratings.append(df)
    
    if not all_ratings:
        print("  [WARNING] No rating files found!")
        return pd.DataFrame()
    
    combined = pd.concat(all_ratings, ignore_index=True)
    print(f"  Total ratings loaded: {len(combined):,}")
    return combined

In [20]:
def main():
    print("=" * 60)
    print("BUILDING MASTER PLAYER DATABASE")
    print("=" * 60)

    # --- A) Load Kaggle data -------------------------------------------
    print("\n[1/5] Loading Kaggle / Transfermarkt data...")
    players     = load_kaggle_players()
    valuations  = load_kaggle_valuations()
    print(f"       Players loaded:    {len(players):,}")
    print(f"       Valuations loaded: {len(valuations):,}")

    # --- B) Load player stats (Big 5 combined files) -------------------
    print("\n[2/5] Loading Big 5 player stats files...")
    player_stats = load_player_stats_files()

    # --- C) Load player ratings (league-specific folders) --------------
    print("\n[3/5] Loading player ratings...")
    ratings = load_player_ratings()

    # --- D) Load & merge all FBref seasons -----------------------------
    print("\n[4/5] Loading FBref season stats...")
    all_seasons = []

    for season in SEASONS:
        print(f"\n  --- {season} ---")
        for league_name, league_code in LEAGUES.items():
            df = build_fbref_season(league_name, league_code, season)
            if not df.empty:
                all_seasons.append(df)

    fbref_all = pd.concat(all_seasons, ignore_index=True)
    print(f"\n       Total FBref rows: {len(fbref_all):,}")

    # --- E) Match FBref players to Kaggle player_ids -------------------
    print("\n[5/5] Matching players across all sources...")
    
    # Normalize names for matching
    players['name_clean'] = players['name'].str.strip().str.lower()
    fbref_all['name_clean'] = fbref_all['name'].str.strip().str.lower()

    # Merge FBref with Kaggle players
    merged = fbref_all.merge(
        players[['player_id', 'name', 'position', 'date_of_birth', 'name_clean']],
        on='name_clean',
        how='left',
        suffixes=('_fbref', '_kaggle')
    )

    # Use Kaggle name where available, fall back to FBref
    merged['name'] = merged['name_kaggle'].fillna(merged['name_fbref'])
    
    # Use Kaggle position if available, fall back to standardised FBref position
    if 'position_kaggle' in merged.columns:
        merged['position'] = merged['position_kaggle'].fillna(
            merged.get('position_fbref', pd.Series(dtype=str))
        )
    elif 'position_fbref' in merged.columns:
        merged['position'] = merged['position_fbref']

    matched   = merged[merged['player_id'].notna()]
    unmatched = merged[merged['player_id'].isna()]
    print(f"       Matched:   {len(matched):,} rows")
    print(f"       Unmatched: {len(unmatched):,} rows")

    # --- F) Attach market valuations ------------------------------------
    master = merged.merge(valuations, on=['player_id', 'season'], how='left')
    print(f"       Rows with market value: {master['market_value_eur'].notna().sum():,}")

    # --- G) Attach player stats -----------------------------------------
    if not player_stats.empty:
        # Normalize names for matching
        player_stats['name_clean'] = player_stats['name'].str.strip().str.lower()
        
        master = master.merge(
            player_stats.drop(columns=['name'], errors='ignore'),
            on=['name_clean', 'season'],
            how='left',
            suffixes=('', '_stats')
        )
        print(f"       Rows with player stats: {master['90s'].notna().sum():,}")

    # --- H) Attach ratings ----------------------------------------------
    if not ratings.empty:
        # Normalize names for matching
        ratings['name_clean'] = ratings['name'].str.strip().str.lower()
        
        master = master.merge(
            ratings[['name_clean', 'season', 'league', 'rating']],
            on=['name_clean', 'season', 'league'],
            how='left'
        )
        print(f"       Rows with ratings: {master['rating'].notna().sum():,}")

    # --- I) Compute derived columns ------------------------------------
    print("\n[6/6] Computing per-90 rates and ages...")
    master = compute_per_90(master)
    master = compute_age_at_season_end(players, master)

    # --- J) Final column order & save -----------------------------------
    final_columns = [
        'fbref_rank',
        'player_id',
        'name',
        'position',
        'season',
        'league',
        'club',
        'club_finish',
        'appearances',
        'minutes',
        'goals',
        'assists',
        'xG',
        'xA',
        'npxG',
        'goals_per_90',
        'assists_per_90',
        'xG_per_90',
        'xA_per_90',
        '90s',                  # NEW - from player stats
        'Gls_90',               # NEW - from player stats
        'Ast_90',               # NEW - from player stats
        'G+A_90',               # NEW - from player stats
        'G-PK_90',              # NEW - from player stats
        'rating',               # NEW - from ratings files
        'market_value_eur',
        'age_at_season_end',
    ]

    # Only keep columns that actually exist
    final_columns = [c for c in final_columns if c in master.columns]
    master = master[final_columns].copy()

    # Sort by player, then season
    master = master.sort_values(['name', 'season']).reset_index(drop=True)

    # Save
    os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
    master.to_csv(OUTPUT_PATH, index=False)

    print("\n" + "=" * 60)
    print(f"DONE — saved {len(master):,} rows to:\n  {OUTPUT_PATH}")
    print("=" * 60)
    print("\nQuick look at the output:\n")
    print(master.head(10).to_string(index=False))
    print(f"\nUnique players: {master['name'].nunique():,}")
    print(f"Seasons covered: {sorted(master['season'].unique())}")
    print(f"Leagues covered: {sorted(master['league'].unique())}")

In [21]:
SEASONS = ['2020-2021', '2021-2022', '2022-2023', '2023-2024', '2024-2025']